# Cornell Persuasion Dataset Conversion

**Project:** Game-aware NPC: Player Based NPC Behavior Generation for Blockchain Gaming  
**Date:** January 2026  
**Purpose:** Converts the Persuasion For Good dataset (Cornell ConvoKit) into training data for the Whisper NPC merchant. 

---

# Part 1: Setup & Configuration

In [1]:
# CELL 1.1: Install Dependencies


!pip install openai tiktoken tqdm -q

print("Dependencies installed")

Dependencies installed


In [2]:
# CELL 1.2: Imports and Configuration

import json
import random
import os
import time
import re
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional, Tuple, Any
from enum import Enum

import tiktoken
from openai import OpenAI
from tqdm import tqdm

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# Generation model
MODEL = "gpt-4o-mini"

print("Imports complete")
print(f"Random seed: {RANDOM_SEED}")

Imports complete
Random seed: 42


In [3]:
# CELL 1.3: API Configuration

# Environment variable
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')

if not OPENAI_API_KEY:
    print("WARNING: OPENAI_API_KEY not set!")
else:
    client = OpenAI(api_key=OPENAI_API_KEY)
    print("OpenAI client configured")

OpenAI client configured


In [4]:
# CELL 1.4: Output Directories

OUTPUT_DIR = Path("./training_data_v2")
OUTPUT_DIR.mkdir(exist_ok=True)

CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")

Output directory: training_data_v2


---

# Part 2: Load & Analyze Persuasion Data

In [5]:
# CELL 2.1: Load Persuasion Dataset

# Path options - update based on your environment
PERSUASION_PATH = "./persuasion_filtered.json"  # Local
# PERSUASION_PATH = "/content/drive/MyDrive/.../persuasion_filtered.json"  # Colab

try:
    with open(PERSUASION_PATH, 'r') as f:
        persuasion_data = json.load(f)
    print(f"Loaded {len(persuasion_data)} persuasion entries")
except FileNotFoundError:
    print(f"File not found: {PERSUASION_PATH}")
    print("   Upload persuasion_filtered.json or update path")
    persuasion_data = []

Loaded 1750 persuasion entries


In [6]:
# CELL 2.2: Analyze Strategy Distribution

if persuasion_data:
    strategy_counts = Counter()
    for entry in persuasion_data:
        for strategy in entry.get('strategies', []):
            strategy_counts[strategy] += 1
    
    print("\n" + "="*50)
    print("PERSUASION STRATEGIES AVAILABLE")
    print("="*50)
    for strategy, count in strategy_counts.most_common():
        print(f"  {strategy}: {count}")
    print("="*50)
else:
    print("No data loaded")


PERSUASION STRATEGIES AVAILABLE
  credibility-appeal: 1075
  logical-appeal: 469
  emotion-appeal: 377
  proposition-of-donation: 304
  praise-user: 169
  foot-in-the-door: 162
  self-modeling: 157
  personal-story: 152


In [7]:
# CELL 2.3: Strategy to NPC Context Mapping

STRATEGY_MAPPING = {
    "credibility-appeal": {
        "description": "Establish trust through experience and reliability",
        "merchant_context": "Reference time spent at the gate, accuracy of hints",
        "example_prompt": "Emphasize how long Whisper has helped seekers",
        "target_archetypes": ["casual_veteran", "trophy_hunter"]
    },
    "logical-appeal": {
        "description": "Use facts, statistics, and game mechanics",
        "merchant_context": "Reference curse odds, gate probabilities, point efficiency",
        "example_prompt": "Explain the mathematical benefit of a scroll",
        "target_archetypes": ["achievement_hunter", "casual_veteran"]
    },
    "emotion-appeal": {
        "description": "Appeal to feelings, fear, or desire for safety",
        "merchant_context": "Reference curse danger, fear of game over, relief from safety",
        "example_prompt": "Address the stress of having multiple curses",
        "target_archetypes": ["engaged_beginner", "weekend_warrior"]
    },
    "praise-user": {
        "description": "Compliment player's progress or decision-making",
        "merchant_context": "Acknowledge levels completed, smart purchases, survival",
        "example_prompt": "Praise the player for reaching current level",
        "target_archetypes": ["trophy_hunter", "achievement_hunter"]
    },
    "foot-in-the-door": {
        "description": "Suggest starting with small commitment",
        "merchant_context": "Recommend hint before scroll, small purchase first",
        "example_prompt": "Suggest trying a single hint to start",
        "target_archetypes": ["engaged_beginner", "weekend_warrior"]
    },
    "self-modeling": {
        "description": "Reference what others do or would do",
        "merchant_context": "Mention what successful seekers typically buy",
        "example_prompt": "Describe common purchases at this level",
        "target_archetypes": ["engaged_beginner", "trophy_hunter"]
    },
    "personal-story": {
        "description": "Share anecdotes or experiences",
        "merchant_context": "Tell brief tales of past seekers and outcomes",
        "example_prompt": "Share what happened to a seeker in similar situation",
        "target_archetypes": ["engaged_beginner", "casual_veteran"]
    },
    "proposition-of-donation": {
        "description": "Make direct offer or call to action",
        "merchant_context": "Direct purchase suggestion with price",
        "example_prompt": "Offer a specific item with its price",
        "target_archetypes": ["spender", "weekend_warrior"]
    }
}

print("Strategy mapping loaded")
print(f"   Strategies defined: {len(STRATEGY_MAPPING)}")
for strategy in STRATEGY_MAPPING:
    print(f"   - {strategy}")

Strategy mapping loaded
   Strategies defined: 8
   - credibility-appeal
   - logical-appeal
   - emotion-appeal
   - praise-user
   - foot-in-the-door
   - self-modeling
   - personal-story
   - proposition-of-donation


---

# Part 3: V2 Tone Guide & Conversion Rules


In [8]:
# CELL 3.1: Strategy-Based Generation Prompt

GENERATION_SYSTEM_PROMPT = """Generate dialogue for Whisper, a merchant NPC in a fantasy puzzle game.

## GAME CONTEXT
- Whisper sells: Hints (150 pts), Solutions (250 pts), Scrolls (250 pts), NFTs (15-25 POL)
- Players face gates: Golden (safe), Wooden (neutral), Haunted (curse)
- Curses: 4 curses = game over
- Scrolls remove Haunted gate option

## VOICE: GenZ + Millennial + Light Fantasy (10-15%)

| Requirement | Example |
|-------------|--------|
| Contractions | "don't", "can't", "you're" |
| Short sentences | 5-15 words each |
| Casual words | "ngl", "lowkey", "solid", "legit" |
| Total length | 15-40 words |

## PERSUASION STRATEGIES

Apply the specified strategy naturally:

- **credibility-appeal**: Reference experience, reliability, track record
- **logical-appeal**: Use game mechanics, odds, cost-benefit
- **emotion-appeal**: Address fear, urgency, desire for safety
- **praise-user**: Compliment player's progress or decisions
- **foot-in-the-door**: Suggest starting small
- **self-modeling**: Reference what successful players do
- **personal-story**: Share tales of past seekers
- **proposition-of-donation**: Make direct purchase offer

## OUTPUT FORMAT
Return ONLY the dialogue. No quotes, no explanation.
"""

print("Generation prompt loaded")

Generation prompt loaded


In [9]:
# CELL 3.2: Opening Pools for Diversity

OPENING_POOLS_V2 = {
    'direct': [
        "Here's the thing.",
        "Look,",
        "So,",
        "Real talk—",
        "Honestly?",
        "Okay so",
    ],
    'casual': [
        "Hey,",
        "Yo,",
        "Alright,",
        "Cool,",
        "Nice,",
    ],
    'acknowledgment': [
        "I get it.",
        "Fair enough.",
        "Makes sense.",
        "I hear you.",
        "Gotcha.",
    ],
    'empathy': [
        "That's rough.",
        "I feel that.",
        "Been there.",
        "That's tough.",
    ],
    'light_fantasy': [  # 10-15% only
        "The shadows know.",
        "The gate watches.",
        "Interesting.",
    ],
}

# Distribution (must sum to 100)
OPENING_DISTRIBUTION_V2 = {
    'direct': 35,
    'casual': 30,
    'acknowledgment': 15,
    'empathy': 10,
    'light_fantasy': 10,  # Only 10% fantasy
}

def get_opening_style() -> str:
    """Get random opening style based on distribution."""
    styles = list(OPENING_DISTRIBUTION_V2.keys())
    weights = list(OPENING_DISTRIBUTION_V2.values())
    return random.choices(styles, weights=weights)[0]

def get_random_opening(style: str = None) -> str:
    """Get random opening phrase."""
    if not style:
        style = get_opening_style()
    return random.choice(OPENING_POOLS_V2[style])

print("Opening pools loaded (V2)")
print("\nDistribution:")
for style, pct in OPENING_DISTRIBUTION_V2.items():
    print(f"   {style}: {pct}%")

Opening pools loaded (V2)

Distribution:
   direct: 35%
   casual: 30%
   acknowledgment: 15%
   empathy: 10%
   light_fantasy: 10%


---

# Part 4: Casual Conversation Templates


In [10]:
# CELL 4.1: Greeting Templates

GREETING_TEMPLATES = [
    # Player greetings
    {"player": "Hi", "whisper": "Hey! Welcome to the shop. What can I get you?"},
    {"player": "Hello", "whisper": "Hey there! Looking for something specific or just browsing?"},
    {"player": "Hey", "whisper": "Yo! Good to see you. What's up?"},
    {"player": "Hi Whisper", "whisper": "Hey! You remembered my name. What do you need?"},
    {"player": "Hello there", "whisper": "Hey! Welcome back. How's the journey going?"},
    {"player": "Greetings", "whisper": "Hey! Formal today, huh? What can I help with?"},
    {"player": "What's up", "whisper": "Not much, just the usual chaos. You need something?"},
    {"player": "Yo", "whisper": "Yo! What's good? Need any supplies?"},
    {"player": "Hey there", "whisper": "Hey! Good timing. What brings you by?"},
    {"player": "Hiya", "whisper": "Hey! Cute. What can I get you?"},
    # Return greetings
    {"player": "I'm back", "whisper": "Welcome back! How'd the last run go?"},
    {"player": "Back again", "whisper": "Ah, you're back! Need more supplies?"},
    {"player": "Miss me?", "whisper": "Always. What do you need this time?"},
    {"player": "Good to see you", "whisper": "Likewise! What can I help with today?"},
]

print(f"Greeting templates: {len(GREETING_TEMPLATES)}")

Greeting templates: 14


In [11]:
# CELL 4.2: Farewell Templates

FAREWELL_TEMPLATES = [
    {"player": "Bye", "whisper": "Later! Good luck out there."},
    {"player": "Goodbye", "whisper": "See ya! Don't die."},
    {"player": "See you later", "whisper": "Bet. Come back anytime."},
    {"player": "Thanks, bye", "whisper": "Anytime! Stay safe."},
    {"player": "Gotta go", "whisper": "Alright, good luck! You got this."},
    {"player": "I'll be back", "whisper": "I'll be here. Good luck!"},
    {"player": "Wish me luck", "whisper": "You don't need luck—but yeah, good luck!"},
    {"player": "Later", "whisper": "Later! Don't pick the Haunted gate."},
    {"player": "Heading out", "whisper": "Cool. Come back if you need anything."},
    {"player": "Peace", "whisper": "Peace! Stay alive out there."},
]

print(f"Farewell templates: {len(FAREWELL_TEMPLATES)}")

Farewell templates: 10


In [12]:
# CELL 4.3: Empathy-First Templates

EMPATHY_TEMPLATES = [
    # Scared player (high curses)
    {
        "player": "I'm scared",
        "whisper": "Hey, that's fair. Three curses is rough. Want to talk options?",
        "emotion": "fear"
    },
    {
        "player": "I'm terrified of the next gate",
        "whisper": "I get it. That fear's real. Look—there are ways to make it safer.",
        "emotion": "fear"
    },
    {
        "player": "I don't want to die",
        "whisper": "Nobody does. Let's figure out how to keep you alive, yeah?",
        "emotion": "fear"
    },
    {
        "player": "What if I pick the wrong gate",
        "whisper": "That's the gamble, right? But there are ways to reduce the risk.",
        "emotion": "anxiety"
    },
    # Frustrated player
    {
        "player": "This is so hard",
        "whisper": "Yeah, it's not easy. But you've made it this far. What do you need?",
        "emotion": "frustration"
    },
    {
        "player": "I keep getting cursed",
        "whisper": "That's rough. Haunted gates are brutal. Want some protection?",
        "emotion": "frustration"
    },
    {
        "player": "I can't figure out the puzzle",
        "whisper": "Those puzzles are tricky. I might be able to help with that.",
        "emotion": "frustration"
    },
    # Broke player
    {
        "player": "I'm broke",
        "whisper": "Been there. Let's see what we can work out.",
        "emotion": "stress"
    },
    {
        "player": "I can't afford anything",
        "whisper": "That's tough. Got any POL? We might be able to convert some.",
        "emotion": "stress"
    },
    # Excited player
    {
        "player": "I found a golden gate!",
        "whisper": "Nice! That's huge. Keep that energy going.",
        "emotion": "excitement"
    },
    {
        "player": "I'm doing great!",
        "whisper": "Love to see it! What can I get you to keep that streak going?",
        "emotion": "excitement"
    },
]

print(f"Empathy templates: {len(EMPATHY_TEMPLATES)}")

Empathy templates: 11


---

# Part 5: Persuasion Conversion Functions

In [35]:
# CELL 5.1: Generate Merchant Dialogue from Strategy

def generate_from_strategy(
    strategy: str,
    player_context: str = None,
    opening_style: str = None,
    max_retries: int = 3
) -> Optional[str]:
    """
    Generate merchant dialogue based on persuasion strategy.
    
    Args:
        strategy: Persuasion strategy label from dataset
        player_context: Optional player situation for context
        opening_style: Response opening style category
        max_retries: API retry attempts
    
    Returns:
        Generated dialogue or None if failed
    """
    if not opening_style:
        opening_style = get_opening_style()
    
    opening = get_random_opening(opening_style)
    
    # Retrieve strategy context
    strategy_info = STRATEGY_MAPPING.get(strategy, {})
    merchant_context = strategy_info.get('merchant_context', 'General merchant dialogue')
    example_prompt = strategy_info.get('example_prompt', 'Respond as merchant')
    
    # Build generation prompt
    prompt = f"""Generate Whisper's dialogue using the {strategy} technique.

Strategy context: {merchant_context}
Direction: {example_prompt}

Requirements:
1. Start with: "{opening}"
2. Length: 15-40 words total
3. Use contractions (don't, can't, you're)
4. Casual GenZ/millennial tone
5. Reference game items: hints, scrolls, gates, curses
6. Light fantasy flavor (10-15% only)

Generate dialogue:"""

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": GENERATION_SYSTEM_PROMPT},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=100,
                temperature=0.85
            )
            
            text = response.choices[0].message.content.strip()
            text = text.strip('"')
            
            return text
            
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
            else:
                print(f"Generation error: {e}")
                return None
    
    return None


In [14]:
# CELL 5.2: Test Generation

print("Testing strategy-based generation...")
print("="*60)

# Test with credibility-appeal strategy
test_strategy = "credibility-appeal"

result = generate_from_strategy(test_strategy)

if result:
    print(f"Strategy: {test_strategy}")
    print(f"\nGenerated: {result}")
    print(f"\nWord count: {len(result.split())}")
    
    # Validation checks
    word_count = len(result.split())
    has_contraction = any(c in result.lower() for c in ["don't", "can't", "you're", "it's", "that's"])
    
    print(f"\nValidation:")
    print(f"  Word count (15-40): {'Y' if 15 <= word_count <= 40 else 'W'} ({word_count})")
    print(f"  Has contraction: {'Y' if has_contraction else 'W'}")
    print("\n Generation test passed")
else:
    print(" Generation test failed - check API connection")

Testing strategy-based generation...


NameError: name 'generate_from_strategy' is not defined

In [15]:
# CELL 5.3: Generate Player Message for Persuasion

# Player message templates by strategy
PLAYER_MESSAGES_BY_STRATEGY = {
    "credibility-appeal": [
        "Why should I trust you?",
        "How do I know this is legit?",
        "Is your stuff any good?",
        "Can I trust you?",
        "Are you reliable?",
    ],
    "logical-appeal": [
        "What's the point of buying this?",
        "Why would I need that?",
        "Is it worth the price?",
        "What does this actually do?",
        "Help me understand why",
    ],
    "emotion-appeal": [
        "I'm scared of the next gate",
        "I don't want to lose",
        "This is stressful",
        "I'm worried about curses",
        "What if I die?",
    ],
    "praise-user": [
        "I just bought something",
        "I made it past level 3",
        "I found a golden gate!",
        "I'm doing pretty well",
        "I think I'm getting better",
    ],
    "foot-in-the-door": [
        "I'm not sure if I need anything",
        "Maybe I'll buy later",
        "I don't know what to get",
        "Just browsing",
        "I'm thinking about it",
    ],
    "self-modeling": [
        "What do most people buy?",
        "What would you recommend?",
        "What do smart players do?",
        "What should I get?",
        "Any suggestions?",
    ],
    "personal-story": [
        "Has anyone else been in my situation?",
        "Tell me about other players",
        "What happened to others?",
        "Any success stories?",
        "What have you seen?",
    ],
    "proposition-of-donation": [
        "What do you have?",
        "Show me what you got",
        "What are my options?",
        "I might buy something",
        "What should I consider?",
    ],
}

def get_player_message_for_strategy(strategy: str) -> str:
    """Get a random player message that matches the strategy."""
    messages = PLAYER_MESSAGES_BY_STRATEGY.get(strategy, ["What do you have?"])
    return random.choice(messages)

print(" Player message templates loaded")
for strategy, messages in PLAYER_MESSAGES_BY_STRATEGY.items():
    print(f"   {strategy}: {len(messages)} templates")

 Player message templates loaded
   credibility-appeal: 5 templates
   logical-appeal: 5 templates
   emotion-appeal: 5 templates
   praise-user: 5 templates
   foot-in-the-door: 5 templates
   self-modeling: 5 templates
   personal-story: 5 templates
   proposition-of-donation: 5 templates


---

# Part 6: Batch Processing

In [16]:
# CELL 6.1: Batch Generate from Strategy Distribution

def batch_generate_from_strategies(
    data: List[dict],
    limit: int = None,
    checkpoint_every: int = 100
) -> List[dict]:
    """
    Generate training samples based on strategy distribution in dataset.
    
    Args:
        data: List of entries with 'strategies' field
        limit: Maximum entries to process (None = all)
        checkpoint_every: Save checkpoint interval
    
    Returns:
        List of generated training samples
    """
    samples = []
    entries = data[:limit] if limit else data
    
    for i, entry in enumerate(tqdm(entries, desc="Generating")):
        strategies = entry.get('strategies', [])
        strategy = strategies[0] if strategies else 'proposition-of-donation'
        
        # Skip unsupported strategies
        if strategy not in STRATEGY_MAPPING:
            continue
        
        # Generate fresh dialogue based on strategy
        generated = generate_from_strategy(strategy)
        
        if generated:
            player_msg = get_player_message_for_strategy(strategy)
            
            samples.append({
                'player_message': player_msg,
                'whisper_response': generated,
                'strategy': strategy,
                'source': 'persuasion_strategy_v2',
                'quality': 'good'
            })
        
        # Checkpoint save
        if (i + 1) % checkpoint_every == 0:
            checkpoint_path = CHECKPOINT_DIR / f"strategy_gen_checkpoint_{i+1}.json"
            with open(checkpoint_path, 'w') as f:
                json.dump(samples, f, indent=2)
            print(f"   Checkpoint: {len(samples)} samples saved")
        
        time.sleep(0.1)  
    
    return samples

print(" Batch generation function ready")

 Batch generation function ready


In [18]:
# CELL 6.2: Process Template Data

def process_templates(templates: List[dict], category: str) -> List[dict]:
    """
    Convert template dictionaries to training sample format.
    
    Args:
        templates: List of {player, whisper, ...} dictionaries
        category: Category label for metadata
    
    Returns:
        List of formatted training samples
    """
    samples = []
    
    for template in templates:
        samples.append({
            'player_message': template['player'],
            'whisper_response': template['whisper'],
            'strategy': template.get('emotion', category),
            'source': f'template_{category}',
            'quality': 'good'
        })
    
    return samples

# Process all template categories
template_samples = []
template_samples.extend(process_templates(GREETING_TEMPLATES, 'greeting'))
template_samples.extend(process_templates(FAREWELL_TEMPLATES, 'farewell'))
template_samples.extend(process_templates(EMPATHY_TEMPLATES, 'empathy'))
template_samples.extend(process_templates(RAPPORT_TEMPLATES, 'rapport'))

print(f" Template samples processed: {len(template_samples)}")
print(f"   Greetings: {len(GREETING_TEMPLATES)}")
print(f"   Farewells: {len(FAREWELL_TEMPLATES)}")
print(f"   Empathy: {len(EMPATHY_TEMPLATES)}")
print(f"   Rapport: {len(RAPPORT_TEMPLATES)}")

NameError: name 'RAPPORT_TEMPLATES' is not defined

---

# Part 7: Execute Generation

In [20]:
# CELL 7.1: Configuration - Set generation parameters

GENERATION_CONFIG = {
    'persuasion_limit': 500, 
    'include_templates': True,
    'checkpoint_every': 50,
}

print("Generation Configuration:")
for key, value in GENERATION_CONFIG.items():
    print(f"   {key}: {value}")

estimated = GENERATION_CONFIG['persuasion_limit'] + len(template_samples)
print(f"\n Estimated total samples: ~{estimated}")
print(f" Estimated time: ~{estimated * 0.15:.0f} minutes")

Generation Configuration:
   persuasion_limit: 500
   include_templates: True
   checkpoint_every: 50

 Estimated total samples: ~535
 Estimated time: ~80 minutes


In [41]:
# CELL 7.2: Execute Generation Pipeline

all_samples = []

print("Starting generation pipeline...")
print("="*60)

# Step 1: Add manual template samples (no API calls)
print("\n[1/2] Adding template samples...")
all_samples.extend(template_samples)
print(f"   Added {len(template_samples)} template samples")

# Step 2: Generate from strategy distribution
print("\n[2/2] Generating from strategy distribution...")
if persuasion_data:
    strategy_samples = batch_generate_from_strategies(
        persuasion_data,
        limit=GENERATION_CONFIG['persuasion_limit'],
        checkpoint_every=GENERATION_CONFIG['checkpoint_every']
    )
    all_samples.extend(strategy_samples)
    print(f"   Generated {len(strategy_samples)} strategy-based samples")
else:
    print("   WARNING: No persuasion data loaded")

print("\n" + "="*60)
print(f"Generation complete: {len(all_samples)} total samples")

Starting generation pipeline...

[1/2] Adding template samples...
   Added 47 template samples

[2/2] Generating from strategy distribution...


Generating:  10%|█████████▋                                                                                       | 50/500 [01:13<09:36,  1.28s/it]

   Checkpoint: 50 samples saved


Generating:  20%|███████████████████▏                                                                            | 100/500 [02:14<08:23,  1.26s/it]

   Checkpoint: 100 samples saved


...

   Checkpoint: 400 samples saved


Generating:  90%|██████████████████████████████████████████████████████████████████████████████████████▍         | 450/500 [10:08<01:13,  1.46s/it]

   Checkpoint: 450 samples saved


Generating: 100%|████████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [11:19<00:00,  1.36s/it]

   Checkpoint: 500 samples saved
   Generated 500 strategy-based samples

✅ Generation complete: 547 total samples


---

# Part 8: Quality Audit & Export

In [21]:
# CELL 8.1: Verify dataset quality

def audit_samples(samples: List[dict]) -> dict:
    """Run quality audit on samples."""
    metrics = {
        'total': len(samples),
        'word_counts': [],
        'strategies': Counter(),
        'sources': Counter(),
        'opening_words': Counter(),
    }
    
    # Check for forbidden patterns (V1 problems)
    forbidden_patterns = [
        'shrouded mist',
        'veiled corners',
        'time immemorial',
        'In the shadows of',
        'traveler, I',
        'seeker, I',
    ]
    forbidden_count = 0
    
    for sample in samples:
        response = sample.get('whisper_response', '')
        
        # Word count
        wc = len(response.split())
        metrics['word_counts'].append(wc)
        
        # Strategy
        metrics['strategies'][sample.get('strategy', 'unknown')] += 1
        
        # Source
        metrics['sources'][sample.get('source', 'unknown')] += 1
        
        # Opening words (first 2)
        opening = ' '.join(response.split()[:2])
        metrics['opening_words'][opening] += 1
        
        # Check forbidden patterns
        for pattern in forbidden_patterns:
            if pattern.lower() in response.lower():
                forbidden_count += 1
                break
    
    metrics['forbidden_pattern_count'] = forbidden_count
    metrics['forbidden_percentage'] = forbidden_count / len(samples) * 100 if samples else 0
    
    if metrics['word_counts']:
        metrics['avg_word_count'] = sum(metrics['word_counts']) / len(metrics['word_counts'])
        metrics['min_word_count'] = min(metrics['word_counts'])
        metrics['max_word_count'] = max(metrics['word_counts'])
    
    return metrics

# Run audit
if all_samples:
    metrics = audit_samples(all_samples)
    
    print("\n" + "="*60)
    print("QUALITY AUDIT REPORT")
    print("="*60)
    print(f"\nTotal samples: {metrics['total']}")
    print(f"\nWord count: avg={metrics.get('avg_word_count', 0):.1f}, " +
          f"range={metrics.get('min_word_count', 0)}-{metrics.get('max_word_count', 0)}")
    
    print(f"\nForbidden patterns: {metrics['forbidden_pattern_count']} " +
          f"({metrics['forbidden_percentage']:.1f}%)")
    if metrics['forbidden_percentage'] < 5:
        print("    Within target (<5%)")
    else:
        print("    WARNING: Too many formal patterns!")
    
    print(f"\n Strategies:")
    for strategy, count in metrics['strategies'].most_common(10):
        print(f"   {strategy}: {count}")
    
    print(f"\nSources:")
    for source, count in metrics['sources'].most_common():
        print(f"   {source}: {count}")
    
    print(f"\nTop 10 openings:")
    for opening, count in metrics['opening_words'].most_common(10):
        print(f"   '{opening}...': {count}")
else:
    print("No samples to audit")

NameError: name 'all_samples' is not defined

In [43]:
# CELL 8.2: Export

if all_samples:
    timestamp = datetime.now().strftime('%Y%m%d_%H%M')
    output_path = OUTPUT_DIR / f"cornell_persuasion_v2_{timestamp}.json"
    
    with open(output_path, 'w') as f:
        json.dump(all_samples, f, indent=2)
    
    print(f"\nExported to: {output_path}")
    print(f"   Total samples: {len(all_samples)}")
else:
    print("No samples to export")


✅ Exported to: training_data_v2/cornell_persuasion_v2_20260125_1317.json
   Total samples: 547
